In [1]:
from modeling_distillemb import BertModel, BertForSequenceClassification, BertForEmbeddingLM
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel
from datasets import load_dataset

In [2]:
num_input_chars=12

In [3]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

In [4]:
distill_config

DistillEmbConfig {
  "activation": "gelu",
  "architectures": [
    "DistillEmb"
  ],
  "char_vocab_size": 1518,
  "distill_dropout": 0.0,
  "dtype": "float32",
  "embedding_size": 512,
  "model_type": "distilemb",
  "num_input_chars": 12,
  "pad_char_id": 0,
  "size": "base",
  "transformers_version": "4.57.1",
  "use_normalize": false,
  "use_tanh": false
}

In [ ]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=512,
    num_hidden_layers=2,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.5,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False
)


In [6]:
# path = "downstream-data/sentiment.parquet"
# df = pd.read_parquet(path)
# if 'sent' in path:
#     # remove 0th index
#     df = df[df['text'] != 'tweet'].reset_index(drop=True)

In [7]:
path = "downstream-data/afrihate.parquet"
import os

if not os.path.exists(path):
    train_dfs = []
    test_dfs = []
    val_dfs = []
    for lang in ['amh', 'arq', 'ary', 'hau', 'ibo', 'kin', 'orm', 'som', 'swa', 'pcm', 'tir', 'twi', 'xho', 'yor', 'zul']:
        ds = load_dataset("afrihate/afrihate", lang)
        tdf = ds['train'].to_pandas()
        tdf['lang'] = lang
        train_dfs.append(tdf)
        tdf = ds['test'].to_pandas()
        tdf['lang'] = lang
        test_dfs.append(tdf)
        tdf = ds['validation'].to_pandas()
        tdf['lang'] = lang
        val_dfs.append(tdf)
    train_df = pd.concat(train_dfs).reset_index(drop=True)
    test_df = pd.concat(test_dfs).reset_index(drop=True)
    val_df = pd.concat(val_dfs).reset_index(drop=True)
    train_df['split'] = 'train'
    test_df['split'] = 'test'
    val_df['split'] = 'val'
    df = pd.concat([train_df, test_df, val_df]).reset_index(drop=True)
    df.to_parquet(path)
else:
    df = pd.read_parquet(path)

In [8]:
import re

def anonymize_and_normalize_text(text: str, lowercase: bool = True) -> str:
    """
    Preprocess text exactly as in AfriSenti[](https://arxiv.org/pdf/2302.08956):
    - Replace all @mentions with '@user'
    - Remove all URLs
    - Optionally lowercase (used for Nigerian languages in the paper)
    - Clean up whitespace
    
    Args:
        text (str): Raw input text
        lowercase (bool): Set to True for Nigerian Pidgin, Hausa, etc.; False for others
    
    Returns:
        str: Cleaned text
    """
    if not isinstance(text, str):
        return text
    
    # 1. Replace @mentions with @user
    text = re.sub(r'@[\w]+', '@user', text)
    
    # 2. Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)                    # http:// or https://
    text = re.sub(r'www\.\S+', '', text)                          # www.
    text = re.sub(r'\b\S+\.(com|org|net|edu|gov)\b', '', text)     # domain.com
    
    # 3. Optional lowercasing (used in AfriSenti for Nigerian languages)
    if lowercase:
        text = text.lower()
    
    # 4. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [9]:
len(df['lang'].unique())

15

In [10]:
lang_counts = df.groupby('split')['lang'].nunique()
for split, count in lang_counts.items():
    print(f"{split.capitalize()} split has {count} languages.")

Test split has 15 languages.
Train split has 15 languages.
Val split has 15 languages.


In [11]:
label2id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df['label'] = df['label'].map(label2id).astype(int)

config.label2id = label2id
config.id2label = id2label

print(f"Converted labels to integers: {label2id}")

Converted labels to integers: {'Abuse': 0, 'Hate': 1, 'Normal': 2}


In [12]:
num_labels = len(df['label'].unique())
config.num_labels = num_labels
model = BertForSequenceClassification(config)

In [13]:
labels = [x.item() for x in df['label'].unique()]
print(labels)
text_col = 'tweet'

[0, 2, 1]


In [14]:
from datasets import Dataset, DatasetDict
import random
import string
df['text'] = df[text_col].apply(anonymize_and_normalize_text)

def add_gibberish_noise(text: str, min_tokens: int = 1, max_tokens: int = 3, min_length: int = 3, max_length: int = 8) -> str:
    """
    Insert random gibberish tokens into the provided text for augmentation.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    base_tokens = text.split()
    gibberish_tokens = [
        "".join(random.choices(string.ascii_lowercase, k=random.randint(min_length, max_length)))
        for _ in range(random.randint(min_tokens, max_tokens))
    ]
    insert_idx = random.randint(0, len(base_tokens))
    augmented_tokens = base_tokens[:insert_idx] + gibberish_tokens + base_tokens[insert_idx:]
    return " ".join(augmented_tokens)

def build_augmented_dataset(dataframe: pd.DataFrame, samples_per_row: int = 1, separator: str = " "):
    sentiment_aliases = {
        "negative": ("negative", "neg", "0"),
        "neutral": ("neutral", "neu", "1"),
        "positive": ("positive", "pos", "2"),
    }

    def canonical_name(label_id: int) -> str:
        label_name = id2label[label_id].lower()
        for canonical, aliases in sentiment_aliases.items():
            if any(alias in label_name for alias in aliases):
                return canonical
        return label_name

    canonical_to_id = {canonical_name(lbl): lbl for lbl in dataframe["label"].unique()}

    def resolve_label(label_a: int, label_b: int) -> int:
        if 0 in (label_a, label_b):
            return 0
        if 1 in (label_a, label_b):
            return 1
        return 2

    augmented_rows = []
    for _, row in dataframe.iterrows():
        base_text, base_label = row["text"], row["label"]
        if random.random() < 0:
            base_text = add_gibberish_noise(base_text, min_tokens=1, max_tokens=2, min_length=3, max_length=6)
        augmented_rows.append({"text": base_text, "label": base_label})

        if samples_per_row < 1:
            continue
        
        sampled = dataframe.sample(n=5, replace=True)
        for _, sampled_row in sampled.iterrows():
            combined_text = f"{base_text}{separator}{sampled_row['text']}"
            if random.random() < 0.2:
                combined_text = add_gibberish_noise(combined_text, min_tokens=1, max_tokens=4, min_length=3, max_length=10)
            if base_label != sampled_row["label"] and (1 not in [base_label, sampled_row["label"]]):
                continue
            combined_label = resolve_label(base_label, sampled_row["label"])
            augmented_rows.append({"text": combined_text, "label": combined_label})

    augmented_df = pd.DataFrame(augmented_rows)
    # remove duplicates on text
    augmented_df = augmented_df.drop_duplicates(subset=['text']).reset_index(drop=True)
    min_count = augmented_df["label"].value_counts().min()
    augmented_df = (
        augmented_df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(n=min_count, random_state=42))
        .reset_index(drop=True)
    )
    return augmented_df

# Assuming df is your dataframe
# Split the data based on the 'split' column
train_df = df[df['split'] == 'train'][['text', 'label']]
test_df = df[df['split'] == 'test'][['text', 'label']]

# aug_train_df = build_augmented_dataset(train_df, samples_per_row=2, separator=" ")

In [15]:
# aug_train_df

In [16]:
# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
train_dataset

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 62466
})

In [17]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=512,
        return_attention_mask=False,
    )

    batch["labels"] = examples["label"]
    return batch



tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/62466 [00:00<?, ? examples/s]

Map:   0%|          | 0/14250 [00:00<?, ? examples/s]

In [18]:
len(train_dataset[0]['text'].split())

18

In [19]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        return batch

data_collator = CustomDataCollator(tokenizer)

In [20]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted', labels=labels)
    f1_macro = f1_score(labels, predictions, average='macro', labels=labels)
    f1_micro = f1_score(labels, predictions, average='micro', labels=labels)
    return {"accuracy": acc, "f1_weighted": f1, "f1_macro": f1_macro, "f1_micro": f1_micro}


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 16

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.1,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.15,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=16,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro,F1 Micro
1,0.844200,0.844699,0.718667,0.725803,0.714352,0.728526
2,0.821400,0.796599,0.741965,0.749735,0.740965,0.750329
3,0.739600,0.789075,0.747509,0.755541,0.746844,0.755945
4,0.718300,0.774395,0.754035,0.761469,0.752892,0.762251
5,0.705000,0.778517,0.760281,0.766799,0.758828,0.767820
6,0.658800,0.789202,0.761404,0.768824,0.760218,0.769587
7,0.656300,0.779835,0.763158,0.770687,0.762474,0.771112
8,0.630600,0.789947,0.762877,0.770206,0.761907,0.770882
9,0.632400,0.793307,0.761614,0.769006,0.761010,0.769452
10,0.626300,0.793329,0.762246,0.769612,0.761599,0.770068


Evaluation results: {'eval_loss': 0.7933981418609619, 'eval_accuracy': 0.7618245614035087, 'eval_f1_weighted': 0.7691745961295112, 'eval_f1_macro': 0.7611705367111601, 'eval_f1_micro': 0.7696351195796048, 'eval_runtime': 16.1019, 'eval_samples_per_second': 884.99, 'eval_steps_per_second': 55.335, 'epoch': 10.0}


In [26]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm").cuda()
model = trainer.model
model.eval()
# Ensure 'language' column exists in df
test_df = df[df['split'] == 'test'][['text', 'label', 'lang']]
languages = test_df['lang'].unique()
per_language_f1 = {}

batch_size = 16

for lang in languages:
    if lang == 'tg' or lang == 'or':
        continue
    lang_df = test_df[test_df['lang'] == lang]
    size = len(lang_df)
    texts = lang_df['text'].tolist()
    labels = lang_df['label'].values
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding='longest',
            truncation=True,
            max_length=512,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            preds.extend(batch_preds)
    f1 = f1_score(labels, preds, average='macro', labels=labels)
    per_language_f1[lang] = (f1, size)

# Print per-language F1
for lang, (f1, size) in per_language_f1.items():
    print(f"Language: {lang}, F1: {f1:.4f}, Size: {size}")
# Average F1
average_f1 = sum(f1 for f1, size in per_language_f1.values()) / len(per_language_f1)
print(f"Average F1 across languages: {average_f1:.4f}")

# special_languages = ["tg", "or"]
# special_f1_scores = {}

# for lang in special_languages:
#     lang_df = test_df[test_df["lang"] == lang]
#     if lang_df.empty:
#         print(f"No samples found for language '{lang}'.")
#         continue

#     lang_texts = lang_df["text"].tolist()
#     lang_labels = lang_df["label"].values
#     lang_preds = []

#     for i in range(0, len(lang_texts), batch_size):
#         batch_texts = lang_texts[i:i + batch_size]
#         tokenized = tokenizer(
#             batch_texts,
#             padding="longest",
#             truncation=True,
#             max_length=512,
#             return_tensors="pt",
#             return_attention_mask=True,
#             padding_side="right"
#         )
#         with torch.no_grad():
#             inputs = {k: v.cuda() for k, v in tokenized.items()}
#             outputs = model(**inputs)
#             batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
#             lang_preds.extend(batch_preds)

#     f1 = f1_score(lang_labels, lang_preds, average="macro", labels=lang_labels)
#     per_language_f1[lang] = f1
#     special_f1_scores[lang] = f1
#     print(f"Language: {lang}, F1: {f1:.4f}")

# if special_f1_scores:
#     special_average_f1 = sum(special_f1_scores.values()) / len(special_f1_scores)
#     print(f"Average F1 for special languages: {special_average_f1:.4f}")
# else:
#     print("No F1 scores computed for the requested languages.")

Language: amh, F1: 0.6082, Size: 747
Language: arq, F1: 0.5620, Size: 323
Language: ary, F1: 0.7758, Size: 699
Language: hau, F1: 0.7343, Size: 1049
Language: ibo, F1: 0.8653, Size: 821
Language: kin, F1: 0.6939, Size: 714
Language: orm, F1: 0.7031, Size: 759
Language: som, F1: 0.6335, Size: 745
Language: swa, F1: 0.8922, Size: 3168
Language: pcm, F1: 0.6571, Size: 1593
Language: tir, F1: 0.6780, Size: 765
Language: twi, F1: 0.7397, Size: 698
Language: xho, F1: 0.7257, Size: 622
Language: yor, F1: 0.7068, Size: 819
Language: zul, F1: 0.7797, Size: 728
Average F1 across languages: 0.7170


In [22]:
# model.save_pretrained("distil-emb-news-lstm-best-256")

In [23]:
numbers = [75.25, 80.76, 63.31, 82.20, 89.85, 79.56, 77.62, 69.20, 72.26, 91.22, 77.55, 78.68, 86.83, 74.32, 86.81]
mean = sum(numbers) / len(numbers)
print(mean)  # Outputs: 78.97375

79.02799999999999


In [24]:
# 69.54 67.93 30.48 82.28 89.53 79.43 73.43 66.90 65.52 91.36 73.07 74.54 81.07 72.37 83.75 
numbers = [69.54, 67.93, 30.48, 82.28, 89.53, 79.43, 73.43, 66.90, 65.52, 91.36, 73.07, 74.54, 81.07, 72.37, 83.75]
mean = sum(numbers) / len(numbers)
print(mean)  # Outputs: 73.252

73.41333333333333
